## סדרה מחושבת: משוואת התנועה כלולאת עדכון

ב-10.1 וב-10.2 הסדרה נמדדה; כאן היא מחושבת. משוואת התנועה נותנת את קצב השינוי, ולכן גם כלל מצעד לצעד: אם $dv/dt = a$, אחרי צעד זמן ‎`dt`‎ המהירות היא $v_{n+1} = v_n + a\,dt$. כל ערך תלוי בקודמו (יחס נסיגה), ולכן לולאה ממלאת את המערך צעד אחר צעד, כמו ‎`simulate`‎ ורשימת ההיסטוריה שלה בחצי הראשון של הקורס.

### דוגמה: זריקה אנכית צעד אחר צעד

כדור נזרק ישר למעלה במהירות 15 מ'/ש'. בלי התנגדות אוויר:

$$v_{n+1} = v_n - g\,dt \qquad y_{n+1} = y_n + v_n\,dt$$

לזריקה הזו יש נוסחה, $y = v_0 t - \tfrac{1}{2} g t^2$, ולכן אפשר לבדוק את הלולאה מולה.

In [ ]:
import numpy as np

g = 9.8
v0 = 15.0
dt = 0.01
n_steps = int(2 * v0 / g / dt) + 1

v_sim = np.zeros(n_steps)          # before the loop: empty arrays
y_sim = np.zeros(n_steps)
v_sim[0] = v0                      # and the starting velocity
for n in range(n_steps - 1):
    v_sim[n + 1] = v_sim[n] - g * dt      # every round: from the previous
    y_sim[n + 1] = y_sim[n] + v_sim[n] * dt

t_sim = np.arange(n_steps) * dt    # after the loop: the time of every step
y_formula = v0 * t_sim - 0.5 * g * t_sim**2
print(round(y_sim.max(), 2), round(y_formula.max(), 2))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.plot(t_sim, y_sim, label="Update loop")
ax.plot(t_sim, y_formula, color="gray", linestyle="--", label="Formula")
ax.set_xlabel("t (s)")
ax.set_ylabel("y (m)")
ax.legend()
plt.show()

השיא של הלולאה גבוה ב-0.07 מטר מזה של הנוסחה, כי הלולאה מניחה שבמשך כל צעד המהירות קבועה. עם ‎`dt`‎ קטן פי 10, ההפרש קטן פי 10.

### מעקב אחרי לולאה בטבלה

כשלולאת עדכון מדפיסה ערכים לא נכונים, הכלי השימושי ביותר הוא טבלה: עמודה ל-‎`i`‎ ועמודה לכל מערך שהלולאה ממלאת. ממלאים אותה שורה אחר שורה, כל שורה מתוך השורה שלפניה, בדיוק כמו המחשב. ניקח לולאה קטנה עם מספרים עגולים: זריקה אנכית במהירות 10 מטר לשנייה, עם ‎`g = 10`‎ ו-‎`dt = 0.1`‎, בארבעה צעדים.

In [ ]:
v = np.zeros(4)      # velocity (m/s)
y = np.zeros(4)      # height (m)
v[0] = 10.0
for i in range(3):
    v[i + 1] = v[i] - 10.0 * 0.1
    y[i + 1] = y[i] + v[i] * 0.1
print(v)
print(y)

| ‎`i`‎ | ‎`v[i]`‎ | ‎`y[i]`‎ | מה הסיבוב הזה מחשב |
|---|---|---|---|
| 0 | 10 | 0 | `v[1] = 10 - 1 = 9`, `y[1] = 0 + 10 * 0.1 = 1` |
| 1 | 9 | 1 | `v[2] = 9 - 1 = 8`, `y[2] = 1 + 9 * 0.1 = 1.9` |
| 2 | 8 | 1.9 | `v[3] = 8 - 1 = 7`, `y[3] = 1.9 + 8 * 0.1 = 2.7` |
| 3 | 7 | 2.7 | אין סיבוב כזה: ‎`range(3)`‎ נעצר לפני 3 |

כך משתמשים בטבלה כשהפלט שגוי:

1. ממלאים אותה לפי מה שהקוד **צריך** לעשות, עוד לפני שמסתכלים על הפלט.
2. משווים לפלט ומחפשים את הערך **הראשון** שיוצא שונה. כל ערך שאחריו כבר חושב מתוך ערך שגוי, ולכן הוא מלמד פחות.
3. השורה בקוד שמחשבת את הערך הזה היא החשודה הראשונה. אם ‎`v`‎ נכון ו-‎`y`‎ שגוי, הבעיה בשורה של ‎`y`‎. אם הלולאה נעצרת ב-‎`IndexError`‎, בודקים מה ה-‎`i`‎ האחרון ש-‎`range`‎ מייצר.

### כמה ריצות בכל צעד

כדי לחשב כמה ריצות בבת אחת, למשל כמה ערכים של קבוע ההתנגדות ‎`k`‎, המערך מקבל ציר ריצות. הלולאה עדיין רצה רק על הזמן, וכל צעד מעדכן עמודה שלמה:

$$v_{n+1} = v_n - (g + k\,v_n)\,dt$$

‎`v[:, n]`‎ היא המהירות בצעד ‎`n`‎ בכל הריצות, בצורה ‎`(3,)`‎, כמו ‎`k`‎, ולכן ‎`k * v[:, n]`‎ היא פעולה איבר-איבר.

In [ ]:
k = np.array([0.0, 0.5, 1.0])      # drag constants (1/s)
n_steps = 401                      # 4 s
v = np.zeros((3, n_steps))         # (k value, step)
y = np.zeros((3, n_steps))
v[:, 0] = v0
for n in range(n_steps - 1):
    v[:, n + 1] = v[:, n] - (g + k * v[:, n]) * dt
    y[:, n + 1] = y[:, n] + v[:, n] * dt

print(v.shape)
print(y.max(axis=1))               # the highest point for every k

In [ ]:
t_k = np.arange(n_steps) * dt
fig, ax = plt.subplots()
for j in range(3):
    ax.plot(t_k, v[j], label=f"k = {k[j]} 1/s")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlabel("t (s)")
ax.set_ylabel("v (m/s)")
ax.legend()
plt.show()

בלי התנגדות המהירות יורדת בקו ישר. עם התנגדות הכדור עולה לגובה נמוך יותר, ובירידה המהירות מתקרבת לערך קבוע, $-g/k$, שבו ההתנגדות מאזנת את הכובד.

### דוגמה נוספת: כדור שוקע בנוזל צמיג

כדור קטן משוחרר ממנוחה בתוך נוזל צמיג. כשהעומק ‎`z`‎ נמדד כלפי מטה, משוואת התנועה שלו היא

$$\frac{dv}{dt} = g' - k\,v$$

כאשר $g'$ היא תאוצת הכובד אחרי ניכוי העילוי, ו-$k$ תלוי בצמיגות. בניסוי הזה $g' = 2.0$ מ'/ש² ו-$k = 8.0$ לשנייה. מצלמה מדדה את העומק 50 פעמים בשנייה, במשך שנייה אחת (‎`t_ball`‎ ו-‎`z_ball`‎, במטרים).

In [ ]:
# "measured" depth (m) of a ball sinking in a liquid, 50 frames/s
rng = np.random.default_rng(0)
t_ball = np.round(np.arange(51) * 0.02, 2)
z_ball = 0.25 * (t_ball - (1 - np.exp(-8.0 * t_ball)) / 8.0)
z_ball = np.round(z_ball + rng.normal(0, 0.0001, t_ball.shape), 4)
z_ball[0] = 0.0

In [ ]:
g_eff, k_ball = 2.0, 8.0                 # m/s^2, 1/s
dt_ball = t_ball[1] - t_ball[0]          # 0.02 s
v_pred = np.zeros(len(t_ball))           # the prediction, from rest
for n in range(len(t_ball) - 1):
    v_pred[n + 1] = v_pred[n] + (g_eff - k_ball * v_pred[n]) * dt_ball

v_meas = np.diff(z_ball) / dt_ball       # the measurement, as in 10.1
t_mid = (t_ball[:-1] + t_ball[1:]) / 2
print(round(v_pred[-1], 3))
print(round(v_meas[t_mid > 0.6].mean(), 3))

In [ ]:
fig, ax = plt.subplots()
ax.plot(t_mid, v_meas, "o", markersize=3, label="Measured (np.diff)")
ax.plot(t_ball, v_pred, label="Prediction (update loop)")
ax.set_xlabel("t (s)")
ax.set_ylabel("v (m/s)")
ax.legend()
plt.show()

התחזית מגיעה למהירות סופית, שבה $dv/dt = 0$, כלומר $v = g'/k = 0.25$ מ'/ש'. הממוצע של המהירות שנמדדה אחרי 0.6 שניות זהה לה עד הספרה השלישית.

### באג נפוץ: ניסיון לוקטר יחס נסיגה

אפשר להתפתות לכתוב ‎`v_wrong[1:] = v_wrong[:-1] - g * dt`‎ כדי לחסוך את הלולאה, אבל זה **לא** עובד:

In [ ]:
v_wrong = np.zeros(len(v_sim))
v_wrong[0] = v0
v_wrong[1:] = v_wrong[:-1] - g * dt    # a wrong attempt to vectorize

print(v_wrong[:5])   # from index 2 on, computed from the old zeros
print(v_sim[:5])     # the right values, from the loop

אגף ימין מחושב **פעם אחת**, מהמערך כפי שהיה לפני השורה: אפסים, חוץ מהאיבר הראשון. לכן רק ‎`v_wrong[1]`‎ יוצא נכון, ומ-‎`v_wrong[2]`‎ והלאה כל האיברים מחושבים מאפסים ישנים ולא מהערכים המעודכנים. בשפה של הטבלה, הערך הראשון ששונה הוא ‎`v_wrong[2]`‎.

### נסו בעצמכם

גוף חם בחדר מתקרר לפי הכלל מהחצי הראשון של הקורס: בכל דקה $T_{n+1} = T_n - k\,(T_n - T_{room})$. השלימו את הפונקציה ‎`cool_steps`‎, שמחזירה מערך של הטמפרטורות בתחילת כל דקה, החל מ-‎`T0`‎. בדקו אותה על הדוגמה שבתא, ואחר כך השוו את הטמפרטורה אחרי 30 דקות לנוסחה מ-8.1, $T_n = 20 + (T_0 - 20)(1-k)^n$.

In [ ]:
def cool_steps(T0, k, T_room, n_steps):
    """Temperature (°C) at the start of every minute, from T0 on."""
    pass   # replace with your code

print(cool_steps(80.0, 0.1, 20.0, 4))
# expected: [80.   74.   68.6  63.74]

`````{admonition} פתרון
:class: dropdown, tip
```python
def cool_steps(T0, k, T_room, n_steps):
    """Temperature (°C) at the start of every minute, from T0 on."""
    T = np.zeros(n_steps)            # before the loop: an empty array
    T[0] = T0                        # and the first value
    for i in range(n_steps - 1):
        T[i + 1] = T[i] - k * (T[i] - T_room)   # from the previous value
    return T                         # after the loop: all the values

print(cool_steps(80.0, 0.1, 20.0, 4))     # [80.   74.   68.6  63.74]
T = cool_steps(80.0, 0.1, 20.0, 31)       # minutes 0 to 30
print(round(T[-1], 3), round(20 + (80.0 - 20) * (1 - 0.1)**30, 3))
```
בסיבוב ‎`i = 0`‎: ‎`80 - 0.1 * (80 - 20) = 74`‎. אחרי 30 דקות הלולאה והנוסחה נותנות 22.543 מעלות: כאן אין שגיאת צעד, כי הכלל עצמו מוגדר לצעד של דקה.
`````

### בדקו את עצמכם

השאלה הראשונה מתייחסת ללולאה הקטנה מהטבלה שלמעלה: ארבעה צעדים, ‎`g = 10`‎ ו-‎`dt = 0.1`‎. השאלות האחרות מתייחסות לקוד הבא:

```python
v = np.zeros(4)
v[0] = 10.0
v[1:] = v[:-1] - 10.0 * 0.1
print(v)                  # line A

k = np.array([0.0, 0.5, 1.0])
v = np.zeros((3, 200))
print(len(v[:, 5]))       # line B
```

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "בלולאה מהטבלה, מה הגובה באינדקס 2 של מערך הגבהים?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "1.9",
                "correct": true,
                "feedback": "נכון: הגובה הקודם, 1, ועוד המהירות הקודמת, 9, כפול 0.1."
            },
            {
                "answer": "1.8",
                "correct": false,
                "feedback": "לא: כאן חושב עם המהירות החדשה, 8. השורה של y משתמשת במהירות של הצעד הקודם."
            },
            {
                "answer": "0.9",
                "correct": false,
                "feedback": "לא: 0.9 הוא רק התוספת של הצעד הזה; צריך להוסיף אותה לגובה הקודם, 1."
            },
            {
                "answer": "2.0",
                "correct": false,
                "feedback": "לא: הגובה לא גדל באותה כמות בכל צעד, כי המהירות קטנה."
            }
        ]
    },
    {
        "question": "כמה מהמספרים ששורה A תדפיס שווים לערכים שהלולאה מהטבלה מחשבת?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "2",
                "correct": true,
                "feedback": "נכון: 10 ו-9. מהאיבר השלישי והלאה אגף ימין חושב מאפסים ישנים."
            },
            {
                "answer": "4",
                "correct": false,
                "feedback": "לא: אגף ימין מחושב פעם אחת, מהמערך הישן, שבו רק האיבר הראשון אינו 0."
            },
            {
                "answer": "1",
                "correct": false,
                "feedback": "לא: גם האיבר השני נכון, כי הוא מחושב מהאיבר הראשון, שכבר היה 10."
            },
            {
                "answer": "3",
                "correct": false,
                "feedback": "לא: האיבר השלישי מחושב מהאיבר השני הישן, 0, ולא מ-9."
            }
        ]
    },
    {
        "question": "מה תדפיס שורה B?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "3",
                "correct": true,
                "feedback": "נכון: זו העמודה של צעד 5, מהירות אחת לכל אחד משלושת ערכי k."
            },
            {
                "answer": "200",
                "correct": false,
                "feedback": "לא: 200 הוא מספר הצעדים, האורך של שורה. כאן נבחרה עמודה."
            },
            {
                "answer": "5",
                "correct": false,
                "feedback": "לא: 5 הוא מספר הצעד שנבחר, לא מספר האיברים."
            },
            {
                "answer": "1",
                "correct": false,
                "feedback": "לא: בעמודה יש ערך אחד לכל ריצה."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)

### תרגול עצמי

השלימו את הפונקציה ‎`drag_velocities`‎, שמחשבת את מהירות הכדור בנוזל לפי $v_{n+1} = v_n + (g' - k\,v_n)\,dt$, ובדקו אותה על הדוגמה שבתא. אחר כך חשבו בעזרתה, עם הנתונים של הדוגמה, את המהירות אחרי שנייה אחת: כשהכדור מתחיל במנוחה, וכשהוא נכנס לנוזל במהירות 0.5 מ'/ש' כלפי מטה. מהי המהירות הסופית בכל מקרה?

In [ ]:
def drag_velocities(v0, g_eff, k, dt, n_steps):
    """Velocity (m/s) at each step, in a liquid with drag k (1/s)."""
    pass   # replace with your code

print(drag_velocities(0.0, 10.0, 1.0, 0.1, 3))
# expected: [0.  1.  1.9]

`````{admonition} פתרון
:class: dropdown, tip
```python
def drag_velocities(v0, g_eff, k, dt, n_steps):
    """Velocity (m/s) at each step, in a liquid with drag k (1/s)."""
    v = np.zeros(n_steps)
    v[0] = v0
    for i in range(n_steps - 1):
        v[i + 1] = v[i] + (g_eff - k * v[i]) * dt
    return v

print(drag_velocities(0.0, 10.0, 1.0, 0.1, 3))   # [0.  1.  1.9]
print(round(drag_velocities(0.0, 2.0, 8.0, 0.02, 51)[-1], 3))   # 0.25
print(round(drag_velocities(0.5, 2.0, 8.0, 0.02, 51)[-1], 3))   # 0.25
```
- בסיבוב ‎`i = 0`‎: ‎`0 + (10 - 1 * 0) * 0.1 = 1`‎
- בסיבוב ‎`i = 1`‎: ‎`1 + (10 - 1 * 1) * 0.1 = 1.9`‎

בשני המקרים המהירות הסופית היא $g'/k = 0.25$ מ'/ש': כשהכדור מהיר ממנה, ההתנגדות גדולה מהכובד והוא מאט.
`````